## 문서 후 처리 
문서 후 처리는 초기 검색 알고리즘으로 검색된 문서들을 대상으로 보다 정교한 방법을 사용하여 문서의 관련성을 재평가하고 순위를 재조정하는 과정을 의미하며 이 과정에서 핵심적인 역할을 하는 것이 리랭킹이다. 

In [40]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_openai.embeddings import OpenAIEmbeddings 
from langchain_qdrant import QdrantVectorStore 
import hashlib

def create_pdf_loader(file_path):
    return PyMuPDFLoader(file_path)


def create_text_splitter(chunk_size: int, chunk_overlap: int):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def get_embedding(model: str):
    return OpenAIEmbeddings(
            model= model,
            base_url="http://localhost:1234/v1",
            api_key="lm-studio",
            check_embedding_ctx_length=False)


def make_doc_id(doc):
    return hashlib.md5(doc.page_content.encode("utf-8")).hexdigest()


def create_qdrant_vector_store_from_documents(docs, embedding, collection_name):
    ids = [make_doc_id(doc) for doc in docs]

    return QdrantVectorStore.from_documents(
        documents=docs,
        embedding=embedding,
        ids=ids,
        url="http://localhost:6333",
        collection_name=collection_name,
    )

In [45]:
file_path = "./data/투자설명서.pdf"

loader = create_pdf_loader(file_path)
doc_splitter = create_text_splitter(300, 100)
docs = loader.load_and_split(doc_splitter)

vectordb = create_qdrant_vector_store_from_documents(docs, get_embedding("bge-m3"), "investment_docs")

In [46]:
from typing import List, Dict, Any, Tuple 
from textwrap import dedent 
from pydantic import BaseModel, Field 
from langchain_core.prompts import PromptTemplate 
from langchain_core.documents import Document 
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI 


In [47]:
class RelevanceScore(BaseModel):
    relevance_score: float = Field(description="문서가 쿼리와 얼마나 관련이 있는지를 나타내는 점수")

def reranking_documents(query: str, doc: List[Document], top_n: int=2) -> List[Document]:
    parser = JsonOutputParser(pydantic_object=RelevanceScore)
    human_message_prompt = PromptTemplate(
        template="""
        1점 부터 10까지의 점수를 매겨, 다음 문서와 질문이 얼마나 관련이 있는지를 평가해주세요.
        단순 키워드가 일치하는 것이 아니라 쿼리의 구체적인 맥락과 의도를 고려하세요. 
        {format_instructions}
        question : {query}
        document : {doc}
        relevance_score:""",
        input_variables=["query", "doc"],
        partial_variables={"format_instructions" : parser.get_format_instructions()}
    )
    
    llm = ChatOpenAI(
        model="gemma-3-4b-it",
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        temperature=0.2, 
        max_tokens=3000)
    
    chain = human_message_prompt | llm | parser 
    
    scored_docs = []
    
    for doc in docs : 
        input_data = {"query": query, "doc" : doc.page_content}
        try:
            score = chain.invoke(input_data)['relevance_score']
            score = float(score)
        except Exception as e :
            print(f"오류 발생 : {str(e)}") 
            default_score = 5 
            print(f"기본 점수 {default_score}점을 사용합니다")
            score = default_score
        scored_docs.append((doc, score))
        
    reranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]

In [48]:
query = "이 회사의 2022년 영업손실이 정확히 얼마야?"

docs = vectordb.similarity_search(
    query=query,
    k=3
)

In [49]:
print(len(docs))
print(docs[1])


3
page_content='업의 성패를 장담할 수 없고, 신규사업 진출이 당사에 악영향을
미치는 결과를 초래할 수 있습니다. 투자자 여러분들께서는 이 점을
반드시 유의하여 주시기 바랍니다.
회사위험
[가. 매출부진 및 지속적 손실 발생 위험]
당사는 매출이 부진한 가운데, 지속적으로 신약후보물질의 발굴과 보유 파
이프라인의 적응증 증가를 위하여 다양한 비임상 및 임상시험을 준비하고
있으며 이에 따라 관련비용의 지출이 꾸준히 발생하여 2021년 영업손실
130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원,' metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': '', 'creationdate': '2024-06-26T16:15:14+09:00', 'source': './data/투자설명서.pdf', 'file_path': './data/투자설명서.pdf', 'total_pages': 514, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-06-26T16:15:14+09:00', 'trapped': '', 'modDate': "D:20240626161514+09'00'", 'creationDate': "D:20240626161514+09'00'", 'page': 11, '_id': '73cfe36a-ee91-01d8-65fe-baad5d5f0133', '_collection_name': 'investment_docs'}


In [50]:
reranked_docs = reranking_documents(query, docs)

In [51]:
print(len(reranked_docs))
print(reranked_docs[1]) 

2
page_content='업의 성패를 장담할 수 없고, 신규사업 진출이 당사에 악영향을
미치는 결과를 초래할 수 있습니다. 투자자 여러분들께서는 이 점을
반드시 유의하여 주시기 바랍니다.
회사위험
[가. 매출부진 및 지속적 손실 발생 위험]
당사는 매출이 부진한 가운데, 지속적으로 신약후보물질의 발굴과 보유 파
이프라인의 적응증 증가를 위하여 다양한 비임상 및 임상시험을 준비하고
있으며 이에 따라 관련비용의 지출이 꾸준히 발생하여 2021년 영업손실
130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원,' metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': '', 'creationdate': '2024-06-26T16:15:14+09:00', 'source': './data/투자설명서.pdf', 'file_path': './data/투자설명서.pdf', 'total_pages': 514, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-06-26T16:15:14+09:00', 'trapped': '', 'modDate': "D:20240626161514+09'00'", 'creationDate': "D:20240626161514+09'00'", 'page': 11, '_id': '73cfe36a-ee91-01d8-65fe-baad5d5f0133', '_collection_name': 'investment_docs'}


In [52]:
for i, doc in enumerate(reranked_docs):
    print(f"\nDocument {i + 1} :")
    print(doc.page_content)


Document 1 :
하여 2021년 영업손실 130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원, 2024년
1분기 영업손실 24.2억원이 발생하였습니다. 또한 영업 외적 측면에서도, 금융비용 등의 발생 영향
으로 인해 2021년 당기순손실 130.7억원, 2022년 당기순손실 228.7억원, 2023년 당기순손실
116.1억원, 2024년 1분기 당기순손실 32.9억원이 발생하는 등 지속적인 적자 구조를 면하지 못하고
있습니다.따라서 당사의 파이프라인에서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루

Document 2 :
업의 성패를 장담할 수 없고, 신규사업 진출이 당사에 악영향을
미치는 결과를 초래할 수 있습니다. 투자자 여러분들께서는 이 점을
반드시 유의하여 주시기 바랍니다.
회사위험
[가. 매출부진 및 지속적 손실 발생 위험]
당사는 매출이 부진한 가운데, 지속적으로 신약후보물질의 발굴과 보유 파
이프라인의 적응증 증가를 위하여 다양한 비임상 및 임상시험을 준비하고
있으며 이에 따라 관련비용의 지출이 꾸준히 발생하여 2021년 영업손실
130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원,


In [59]:
from langchain_core.retrievers import BaseRetriever 
from langchain_classic.chains import create_retrieval_chain
from pydantic import BaseModel, ConfigDict

class CustomRetriever(BaseRetriever, BaseModel):
    vectorstore : Any = Field(description="Retrivald을 위한 벡터 저장소")
    
    model_config = ConfigDict(
        arbitrary_types_allowed=True
    )
    
    def _get_relevant_documents(self, query: str, num_docs=2) -> List[Document]:
        initial_docs = self.vectorstore.similarity_search(query, k=4)
        return reranking_documents(query, initial_docs, top_n=num_docs)

In [61]:
custom_retriever = CustomRetriever(vectorstore=vectordb)

llm = ChatOpenAI(
        model="gemma-3-4b-it",
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        temperature=0.2)


In [62]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
    ("system", """
당신은 문서 기반 QA assistant입니다.
반드시 제공된 context만 사용해서 답변하세요.
모르면 모른다고 답하세요.

context:
{context}
"""),
    ("human", "{input}")
])

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

retrieval_chain = create_retrieval_chain(
    retriever=custom_retriever,
    combine_docs_chain=document_chain
)

In [64]:
result = retrieval_chain.invoke({"input": "이 회사의 2022년 영업손실은 정확히 얼마야"})
print(result["answer"])
print(result["context"])

2022년 영업손실은 149.1억원이었습니다.
[Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': '', 'creationdate': '2024-06-26T16:15:14+09:00', 'source': './data/투자설명서.pdf', 'file_path': './data/투자설명서.pdf', 'total_pages': 514, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-06-26T16:15:14+09:00', 'trapped': '', 'modDate': "D:20240626161514+09'00'", 'creationDate': "D:20240626161514+09'00'", 'page': 158, '_id': '5ee8c594-6e5a-3706-4083-c0ca19df5f4e', '_collection_name': 'investment_docs'}, page_content='하여 2021년 영업손실 130.1억원, 2022년 영업손실 149.1억원, 2023년 영업손실 122억원, 2024년\n1분기 영업손실 24.2억원이 발생하였습니다. 또한 영업 외적 측면에서도, 금융비용 등의 발생 영향\n으로 인해 2021년 당기순손실 130.7억원, 2022년 당기순손실 228.7억원, 2023년 당기순손실\n116.1억원, 2024년 1분기 당기순손실 32.9억원이 발생하는 등 지속적인 적자 구조를 면하지 못하고\n있습니다.따라서 당사의 파이프라인에서 임상 성공을 통한 기술이전, 상품화 성공 등의 성과를 이루'), Document(metadata={'producer': 'iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': '', 'creat